# Forecasting SPY: Part 1 & Part 2

**Goal:** Use valuation signals (dividend yield DP, earnings yield EP, etc.) to predict next month's SPY return, then turn those predictions into trading strategies and see how well they work.

- **Part 1 (In-Sample):** Fit the regression on *all* the data, then use it to "forecast." This is like peeking at the answer key, so the results are too optimistic.
- **Part 2 (Out-of-Sample):** Each month, estimate the model using *only data available at that time*. This is what you could actually have done in real life.

**Timing convention (important):** The signal at the end of month t predicts the return in month t+1:

$$r_{t+1} = \alpha + \beta' x_t + \varepsilon_{t+1}$$

In code, we shift the signals forward by one month (`.shift(1)`), so each row pairs "last month's signal" with "this month's return." Each forecast is dated by the month it is forecasting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.float_format', '{:.4f}'.format)
FREQ = 12        # monthly data: 12 periods per year
BURN_YRS = 5     # use 5 years of data as a "warm-up" before the first OOS forecast

## 0. Load the Data

The file `spy_forecasting_data.xlsx` has three sheets we need:
- `spy returns`: monthly SPY total returns
- `risk-free rate`: 3-month T-bill rate (monthly), used for the Sharpe ratio
- `signals`: valuation / macro signals

In [ ]:
FILE = 'spy_forecasting_data.xlsx'
spy     = pd.read_excel(FILE, sheet_name='spy returns',    index_col='date', parse_dates=True)['SPY']
rf      = pd.read_excel(FILE, sheet_name='risk-free rate', index_col='date', parse_dates=True)['T3M']
signals = pd.read_excel(FILE, sheet_name='signals',        index_col='date', parse_dates=True)

print('Sample:', spy.index[0].date(), 'to', spy.index[-1].date(), '-', len(spy), 'months')
signals.head()

### Helper Functions

A few functions we will reuse:
1. `ols_fit` / `ols_predict`: ordinary least squares (OLS) regression, returning the intercept and slopes.
2. `perf_stats`: annualized mean, annualized volatility, and Sharpe ratio.

**Sharpe ratio** = annualized excess return (strategy return − risk-free rate) ÷ annualized volatility. Higher means more return per unit of risk.

In [ ]:
def ols_fit(y, X):
    '''OLS regression y = a + X b. Returns the coefficient vector [a, b1, b2, ...]'''
    X = np.column_stack([np.ones(len(X)), np.asarray(X)])   # add a column of 1s for the intercept
    coef, *_ = np.linalg.lstsq(X, np.asarray(y), rcond=None)
    return coef

def ols_predict(coef, X):
    X = np.column_stack([np.ones(len(X)), np.asarray(X)])
    return X @ coef

def perf_stats(rets, rf, freq=FREQ):
    '''Annualized mean, volatility, and Sharpe ratio (based on excess returns)'''
    out = pd.DataFrame(index=rets.columns)
    out['Mean'] = rets.mean() * freq
    out['Vol'] = rets.std() * np.sqrt(freq)
    excess = rets.sub(rf.reindex(rets.index), axis=0)
    out['Sharpe'] = excess.mean() * freq / (rets.std() * np.sqrt(freq))
    return out

def plot_cumulative(rets, title):
    (1 + rets).cumprod().plot(logy=True, figsize=(10, 5), title=title)
    plt.ylabel('Growth of $1 (log scale)')
    plt.grid(alpha=0.3)
    plt.show()

### Aligning the Data: Last Month's Signal ↔ This Month's Return

`signals.shift(1)` moves every signal down one row. After the shift, the signal in the 1993-03-31 row is actually the value observed at the end of 1993-02-28, which is exactly what we would use to predict March's return. The first row has no "previous month," so we drop it with `dropna()`.

In [ ]:
X_lag = signals.shift(1)                       # last month's signals
data = pd.concat([spy, X_lag], axis=1).dropna()
y = data['SPY']                                # this month's SPY return
X = data[signals.columns]                      # last month's signals

MODELS = {                                      # which signals each model uses
    'DP':  ['DP'],
    'EP':  ['EP'],
    'ALL': list(signals.columns),
}
data.head()

---
# Part 1: In-Sample Forecasts

## 1.1 Build Four Forecasts

| Model | How it works |
|---|---|
| **Mean** | Uses no signals. The forecast for next month is just the average of all past monthly returns (expanding mean, shifted one period so it never uses the current month) |
| **DP** | Univariate regression on the dividend yield |
| **EP** | Univariate regression on the earnings yield |
| **ALL** | Multivariate regression on all signals together |

"In-sample" means the DP / EP / ALL coefficients are estimated once using the **entire** sample, and then those coefficients are used to compute every month's forecast.

In [ ]:
fcst_is = pd.DataFrame(index=y.index)

# Mean: historical average return up to last month
fcst_is['Mean'] = spy.expanding().mean().shift(1).reindex(y.index)

# Regression models: estimate coefficients once on the full sample
coefs_is = {}
for name, cols in MODELS.items():
    coef = ols_fit(y, X[cols])
    coefs_is[name] = coef
    fcst_is[name] = ols_predict(coef, X[cols])

fcst_is = fcst_is.dropna()
print('In-sample coefficients (first value is the intercept):')
for k, v in coefs_is.items():
    print(f'  {k:4s}:', np.round(v, 5))

In [ ]:
ax = fcst_is.plot(figsize=(11, 5), title='In-Sample Forecasts: Expected SPY Return Next Month')
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('Monthly forecast')
plt.grid(alpha=0.3); plt.show()

**How to read this chart:**
- **Mean** is nearly flat, because the historical average changes very slowly.
- **DP / EP** move with valuations: when dividend/earnings yields are high (stocks are cheap), the forecast return is higher.
- **ALL** swings the most. With many variables, the regression fits more detail, and more of that detail is likely noise.

## 1.2 Turn Forecasts into Strategies

The rule is simple: **weight = 100 × forecast**.

- If the model predicts +1% (0.01) next month, hold 100 × 0.01 = 1x SPY.
- If it predicts +2%, lever up to 2x. If it predicts a negative return, go short.

Strategy return = weight × next month's SPY return. Since each forecast is already dated by the month it forecasts, we can multiply directly.

We also add a **Passive** benchmark: weight is always 1, i.e., just buy and hold SPY.

In [ ]:
def build_strategies(fcst, spy):
    weights = 100 * fcst
    strat = weights.mul(spy.reindex(fcst.index), axis=0)
    strat['Passive'] = spy.reindex(fcst.index)
    return strat

strat_is = build_strategies(fcst_is, spy)
print('Average weight:'); print((100 * fcst_is).mean().round(2))
perf_stats(strat_is, rf)

In [ ]:
print('Correlation of strategy returns:')
strat_is.corr()

In [ ]:
plot_cumulative(strat_is, 'In-Sample Strategies: Cumulative Return (log scale)')

## 1.3 Interpreting the In-Sample Results

From the output above:

1. **Which signal is most useful?** In-sample, **ALL** has the highest Sharpe (about 0.73), and DP (about 0.62) is slightly better than EP (about 0.59). But ALL is also far more volatile (about 27% vs. 15% for SPY), because its weight swings a lot.
2. **How do they compare to Passive?** Passive has a Sharpe of about 0.60. DP and ALL beat it, EP roughly ties, and Mean is lower. Keep in mind this involves hindsight: the coefficients were estimated using the whole history, including the future.
3. **Are the strategies highly correlated?** Yes. The average weight is about 0.94 for every model, so most of the time they are all long SPY. DP and EP each have a correlation above 0.9 with Passive, and about 0.95 with each other (both are valuation ratios). ALL is less correlated (about 0.6–0.7) because its weight changes the most.

⚠️ **Key caveat:** Good in-sample results should not be taken at face value. In Part 2 we test what would have happened if we had actually traded this way in real time.

---
# Part 2: Out-of-Sample Forecasts

## 2.1 Expanding Window

At the end of each month t:
1. Estimate the regression using **only data through t** (signals through t−1, returns through t).
2. Use the signal at the end of month t to forecast the return in month t+1.
3. Next month, add the new data point, re-estimate, and repeat.

The first **5 years** (60 months) are only a "burn-in" period with no forecasts, since regression coefficients are very unstable when there is little data.

The **Mean** model only ever uses past data (an expanding average), so it is naturally out-of-sample. It is also the **null benchmark** we compare the other models against.

In [ ]:
burn = BURN_YRS * FREQ
dates = y.index                          # each row: last month's signal -> this month's return
fcst_oos = pd.DataFrame(index=signals.index, columns=['Mean'] + list(MODELS), dtype=float)

for i in range(burn, len(dates)):
    t = dates[i]                         # standing at the end of month t
    y_train = y.iloc[:i + 1]             # returns known through t
    X_train = X.iloc[:i + 1]             # matching (lagged) signals
    x_now = signals.loc[[t]]             # latest signal at end of t, used to forecast t+1

    fcst_oos.loc[t, 'Mean'] = y_train.mean()
    for name, cols in MODELS.items():
        coef = ols_fit(y_train, X_train[cols])
        fcst_oos.loc[t, name] = ols_predict(coef, x_now[cols])[0]

# Forecasts made at t are for t+1: shift forward one period so each is dated by the month it forecasts
fcst_oos = fcst_oos.shift(1).dropna()
print('OOS forecast period:', fcst_oos.index[0].date(), 'to', fcst_oos.index[-1].date())
fcst_oos.head()

In [ ]:
ax = fcst_oos.plot(figsize=(11, 5), title='Out-of-Sample Forecasts: Expected SPY Return Next Month')
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('Monthly forecast')
plt.grid(alpha=0.3); plt.show()

**Compared with the in-sample chart:** The OOS forecasts are jumpier, especially early on when there is little data. Because the coefficients are re-estimated every month, they drift over time.

## 2.2 Out-of-Sample Strategy Performance and OOS $R^2$

### What is OOS $R^2$?

$$R^2_{OOS} = 1 - \frac{\text{MSE}_{\text{model}}}{\text{MSE}_{\text{Mean}}}$$

- MSE = the average squared forecast error.
- It compares the model with the simplest possible approach: "just use the historical average."
- **$R^2_{OOS} > 0$**: the model forecasts better than the historical mean ✅
- **$R^2_{OOS} < 0$**: the model does worse than just using the historical mean ❌

In [ ]:
strat_oos = build_strategies(fcst_oos, spy)
perf_stats(strat_oos, rf)

In [ ]:
actual = spy.reindex(fcst_oos.index)
mse = ((fcst_oos.sub(actual, axis=0)) ** 2).mean()
r2_oos = 1 - mse / mse['Mean']
r2_oos.drop('Mean').to_frame('R2_OOS')

In [ ]:
print('Correlation of OOS strategy returns:')
strat_oos.corr()

In [ ]:
plot_cumulative(strat_oos, 'Out-of-Sample Strategies: Cumulative Return (log scale)')

## 2.3 In-Sample vs. Out-of-Sample, Side by Side

For a fair comparison, we restrict the in-sample results to the **same time period** as the OOS results.

In [ ]:
common = strat_oos.index
compare = pd.DataFrame({
    'Sharpe IS': perf_stats(strat_is.loc[common], rf)['Sharpe'],
    'Sharpe OOS': perf_stats(strat_oos, rf)['Sharpe'],
})

# In-sample R2, also relative to the Mean benchmark, for comparison
act_is = spy.reindex(fcst_is.index)
mse_is = ((fcst_is.sub(act_is, axis=0)) ** 2).mean()
compare['R2 IS (vs Mean)'] = (1 - mse_is / mse_is['Mean'])
compare['R2 OOS (vs Mean)'] = r2_oos
compare

In [ ]:
compare[['Sharpe IS', 'Sharpe OOS']].plot.bar(figsize=(8, 4), rot=0,
                                              title='Sharpe Ratio: In-Sample vs. Out-of-Sample')
plt.axhline(0, color='k', lw=0.8); plt.grid(alpha=0.3, axis='y'); plt.show()

### Conclusions (in Plain English)

1. **Do the signals that looked good in-sample still work out-of-sample?** No. Over the same period (1998 onward), DP's Sharpe falls from about 0.49 to 0.34, EP's from about 0.45 to 0.35, and ALL drops the most, from about 0.61 to 0.35. The in-sample regression "knows the future," so its coefficients fit history neatly. Out-of-sample, each month can only use past data, and performance falls apart.
2. **Is OOS $R^2$ positive for any model?** **No, none of them.** DP is about −0.6%, EP about −0.5%, and ALL about −6%. In other words, **every model forecasts worse than simply using the historical average return.** ALL, with the most variables, is the worst. This is classic **overfitting**: the more variables you add, the easier it is to mistake noise for a real pattern.
3. **At the strategy level:** All OOS timing strategies have Sharpe ratios (about 0.34–0.39) **below Passive** (about 0.49). ALL has the highest average return (about 14% per year), but its volatility is about 36%, so it is not worth it on a risk-adjusted basis.
4. **Bottom line:** Valuation signals like DP and EP are related to long-run future returns, but they are very hard to use for **month-by-month** market timing. Any forecasting model must be judged on its **out-of-sample** performance; impressive in-sample results are easily misleading.